# Combining Data Frames



## Setup

In [ ]:
!pip install -U faker


In [ ]:
import pandas as pd
from faker import Faker


In [ ]:
# Initialize Faker
Faker.seed(42)
fake = Faker('en_US')
type(fake)


In [ ]:
# Number of entries you want
num_entries = 6
num_entries


In [ ]:
column_generators = {
            'ID' : lambda: None,
    'First Name' : fake.first_name,
   'Middle Name' : lambda: fake.first_name() if fake.boolean(chance_of_getting_true=75) else None,
     'Last Name' : fake.last_name,
       'Address' : lambda: fake.address().replace('\n', ', '),
  'Phone_Number' : fake.phone_number,
         'Title' : fake.job,
       'Company' : fake.company,
}
column_generators


In [ ]:
df = pd.DataFrame( {
  col_name: [ generator() for _ in range(num_entries) ]
    for col_name, generator in column_generators.items()
})
df["ID"] = (df.index + 1000).astype('Int64')
df


In [ ]:
df.info()

In [ ]:
personal_df = df[
 ['ID',
 'First Name',
 'Middle Name',
 'Last Name',
 'Address',
 'Phone_Number',
 ]
][:4]
personal_df


In [ ]:
work_df = df[
 ['ID',
 'First Name',
 'Middle Name',
 'Last Name',
 'Title',
 'Company']
][2:].assign(_=lambda x: x.index + 100).set_index('_')
work_df


## .concat(), .join(), .merge()

The three pandas functions —`.concat()`, `.join()`, and `.merge()`— combine data frames, but differ primarily in their **purpose** (stacking vs. lookup) and the **axes** (row labels/index vs. columns) they use for alignment.

When specifying the data frame, the first data frame is termed the "left" and the second data frame is termed the "right".

### Summary of Differences

| Function | Primary Role | Default Alignment Key | Flexibility/Control |
| :--- | :--- | :--- | :--- |
| **`pd.concat()`** | Stacking/Appending | Opposite Axis (Index or Columns) | Good for simple structural combination. |
| **`.join()`** | Index Lookup | **Index** | Simple and fast for index-to-index lookup. |
| **`pd.merge()`** | Database-Style Join | **Specific Columns** (Keys) | Highest control over keys and join type. |

## Concatenate

Using the `pd.concat()` method.




### Defaults


#### axis
By default, data frames are stacked one on top of each other, i.e. `axis = 0`.  Where column names match, those columns are used.  Where they do not match, the new columns are added to the right.



In [ ]:
pd.concat([personal_df, work_df])


### Side-by-side

To have data frames extend, use `axis = 1`.  The columns from the right data frame are added to the right.  By default rows are matched on index name.  If there is no match, new rows are added to the bottom.  Column values for rows that don't match are filled in with nulls.


In [ ]:
pd.concat([personal_df, work_df], axis=1)

### Ignoring the row index

This doesn't seem to work.

In [ ]:
pd.concat([personal_df, work_df], ignore_index=True , axis=1)

Resetting the index so that the index of the right matches the index on the left extends the rows.  Notice that non-matching rows from either data frame have their columns filled with nulls.  Also notice that column names do not change, i.e. we now have duplicated column names.


In [ ]:
pd.concat(
  [personal_df,
    work_df
    .reset_index(drop=True)
    .assign(_=lambda x: x.index + 1)
    .set_index('_')
  ],
  axis=1
)


In [ ]:
foo = pd.concat([personal_df.sample(n=2).reset_index(drop=True), work_df.sample(n=2).reset_index(drop=True)], axis = 1)
foo


In [ ]:
foo.columns

In [ ]:
foo.iloc[:, 6]


## Join

The `.join()` method combines two tables based on row index.  When columns have the same name, columns from one of the data frame need different suffixes. Non-matching rows are excluded, even though columns are included.



In [ ]:
personal_df.join(work_df, rsuffix="_r")


In [ ]:
personal_df.join(work_df.reset_index(drop=True), rsuffix="_r")


Often columns are converted to indices.


In [ ]:
(
personal_df
  .set_index("ID")
  .join(
    work_df
    .set_index("ID"),
  rsuffix="_r")
).reset_index()


## Merge



The `.merge()` method combines two tables using column names.

**Purpose:** The most powerful and flexible method, designed for **database-style joins** or lookups. It provides granular control over which columns are used for alignment and what type of join is performed.

| Feature | Description |
| :--- | :--- |
| **Default Axis** | **Column to Column** (or Column to Index) |
| **Alignment** | Aligns one or more **specific columns** (or the index) from one DataFrame with columns/index of the other. |
| **Use Case** | Complex joins based on common ID columns (e.g., joining an `Employees` table on `manager_id` to a `Departments` table on `department_id`). |



### Key Behaviors


* **Explicit Keys:** Requires you to specify the join key(s) using the `on`, `left_on`, `right_on`, `left_index`, and `right_index` parameters.

| Parameter Value (`how`) | Join Type | Logic and Result |
| :--- | :--- | :--- |
| **`cross`** | **Cartesian Product** | Generates a new DataFrame containing **all possible combinations** of rows. Every row from the left DataFrame is combined with every row from the right DataFrame. The keys specified by `on`, `left_on`, etc., are ignored. |
| **`inner`** (Default) | Inner Join | Keeps only records that have **matching keys** in **both** DataFrames. Rows that don't match on the key columns are discarded. |
| **`left`** | Left Outer Join | Keeps **all records** from the **left** DataFrame and the matching records from the right. Where there is no match in the right, columns are filled with missing values. |
| **`right`** | Right Outer Join | Keeps **all records** from the **right** DataFrame and the matching records from the left. Where there is no match in the left, columns are filled with missing values. |
| **`outer`** | Full Outer Join | Keeps **all records** when there is a match in one or both DataFrames. Missing matches in either DataFrame are filled with missing values. |





### how="cross" - Cross or Cartesian Join

This pairs ALL rows from the left data frame with ALL rows from the right data frame. This results in a table of row size p x q, where p is the number of rows in the left data frame and q is the number of rows in the right data frame.

#### Cross or Cartesian Join Example


In [ ]:
personal_df.merge(work_df, how="cross")


In [ ]:
personal_df.merge(work_df, how="cross").shape

In [ ]:
personal_df.shape, work_df.shape

Very inefficient, but works.

In [ ]:
(
personal_df
  .merge(work_df,how="cross")
  .query("ID_x == ID_y", engine='python')
)

### how="inner" - Inner Join




#### Inner Join Example

Using all column names that match.

In [ ]:
personal_df.merge(work_df, how="inner")


Using only one column.  Notice the renaming of columns with the same name.

In [ ]:
personal_df.merge( work_df, how="inner", on=["ID"] )


In [ ]:
personal_df.merge( work_df[["ID","Title","Company"]], how="inner" )


Using indices.

In [ ]:
display(personal_df)
display(work_df)

In [ ]:
personal_df.merge(work_df,how="inner", left_index=True, right_index=True)


In [ ]:
personal_df.merge(work_df.reset_index(drop=True),how="inner", left_index=True, right_index=True)


In [ ]:
personal_df.merge( work_df, how="inner", on=["ID","First Name"] )


### how="left" - Left Outer Join

Similar to `.join()` method.

#### Left Outer Join Example

In [ ]:
personal_df.merge(work_df, how="left", on="ID")


In [ ]:
personal_df.merge(work_df, how="left")


### how="right" - Right Outer Join


#### Right Outer Join Example

In [ ]:
personal_df.merge(work_df,how="right", on="ID")


In [ ]:
work_df.merge(personal_df,how="left", on="ID")


### how="outer" - Full Outer Join

Similar to `.concat()` method.


#### Full Outer Join Example

In [ ]:
personal_df.merge(work_df,how="outer", on="ID")


In [ ]:
foo = personal_df.merge(work_df,how="outer", on="ID")
filter = foo['First Name_y'].isna()
foo[ filter ]


In [ ]:
foo = personal_df.merge(work_df,how="outer", on="ID")
filter = foo['First Name_x'].isna()
foo[ filter ]
